# Transformer Research Notebook

This notebook is the new orchestration surface for the transformer work in `mlworkshop`.

It keeps the workflow notebook-first while moving reusable logic into light local helper modules under `transformer_lab/`.

## What changed

- modern dense baseline: pre-norm, RMSNorm, RoPE, SwiGLU, GQA
- optional sparse MoE path on the same scaffold
- explicit stage plan: base pretrain, domain adapt, SFT, reward refine
- repo-aware local corpus recipe instead of an empty curated-document list
- static FSDP worker module instead of notebook-emitted source blobs
- Plotly-based systems dashboards and richer diagnostics

## Primary files

- `transformer_review.md`
- `transformer_experiment_plan.md`
- `transformer_lab/modeling.py`
- `transformer_lab/training.py`
- `transformer_lab/posttrain.py`
- `transformer_lab/dashboards.py`
- `transformer_lab/fsdp_worker.py`

In [ ]:
# Environment, paths, and shared imports

from pathlib import Path
import json
import math
import random

import numpy as np
import plotly.io as pio
import plotly.graph_objects as go
import sympy as sp
import ipywidgets as widgets
from IPython.display import Markdown, display

import torch

from transformer_lab import (
    CorpusRecipe,
    FsdpLaunchConfig,
    ModelConfig,
    ResearchTransformer,
    TrainConfig,
    build_attention_probe_figure,
    build_dashboard_bundle,
    build_default_eval_suite,
    build_moe_dashboard,
    build_next_token_figure,
    build_repo_instruction_set,
    build_repo_local_corpus,
    build_stage_plan,
    build_training_overview,
    calibration_bins,
    contamination_report,
    count_parameters,
    estimate_kv_cache_bytes,
    load_metrics_rows,
    outcome_reward,
    pack_instruction_batch,
    set_seed,
    write_launch_bundle,
)

pio.renderers.default = "notebook_connected"
set_seed(42)

PROJECT_ROOT = Path.cwd()
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts_transformer"
ARTIFACT_ROOT.mkdir(exist_ok=True)
LAUNCH_ROOT = ARTIFACT_ROOT / "launch"
LAUNCH_ROOT.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Artifact root:", ARTIFACT_ROOT)

In [ ]:
# Read the written review docs inside the notebook

review_path = PROJECT_ROOT / "transformer_review.md"
plan_path = PROJECT_ROOT / "transformer_experiment_plan.md"

if review_path.exists():
    display(Markdown(review_path.read_text(encoding="utf-8")))
if plan_path.exists():
    display(Markdown(plan_path.read_text(encoding="utf-8")))

In [ ]:
# Build a repo-aware local corpus inventory

corpus_recipe = CorpusRecipe()
repo_documents = build_repo_local_corpus(PROJECT_ROOT)

label_counts = {}
for doc in repo_documents:
    label_counts[doc["label"]] = label_counts.get(doc["label"], 0) + 1

print(f"Collected {len(repo_documents)} local documents/chunks from the repo.")
print(json.dumps(label_counts, indent=2))

preview_docs = repo_documents[:5]
for doc in preview_docs:
    print("=" * 100)
    print(doc["id"], "|", doc["label"])
    print(doc["text"][:600])

In [ ]:
# Build phase mixes and contamination diagnostics from the repo-local corpus

from transformer_lab.data import build_phase_documents, weighted_phase_lines, recipe_to_jsonable

phase_line_map = {}
for phase in corpus_recipe.phases:
    grouped = build_phase_documents(repo_documents, phase)
    phase_line_map[phase.name] = weighted_phase_lines(grouped, phase)

for phase_name, lines in phase_line_map.items():
    print(f"{phase_name:12s} -> {len(lines):6d} unique weighted lines")

base_lines = phase_line_map["base"]
adapt_lines = phase_line_map["adapt"]
report = contamination_report(base_lines, adapt_lines)
print("\nBase/adapt overlap report")
print(json.dumps(report, indent=2))

recipe_manifest = recipe_to_jsonable(corpus_recipe)
recipe_manifest_path = ARTIFACT_ROOT / "corpus_recipe.json"
recipe_manifest_path.write_text(json.dumps(recipe_manifest, indent=2), encoding="utf-8")
print("\nSaved corpus recipe manifest:", recipe_manifest_path)


In [ ]:
# Dense and MoE model candidates

dense_cfg = ModelConfig(
    vocab_size=32000,
    max_seq_len=2048,
    d_model=768,
    n_heads=12,
    n_kv_heads=4,
    n_layers=18,
    mlp_ratio=3.5,
    dropout=0.0,
    attn_dropout=0.0,
    resid_dropout=0.0,
    qk_norm=True,
)

moe_cfg = ModelConfig(
    vocab_size=32000,
    max_seq_len=2048,
    d_model=768,
    n_heads=12,
    n_kv_heads=4,
    n_layers=18,
    mlp_ratio=3.5,
    dropout=0.0,
    attn_dropout=0.0,
    resid_dropout=0.0,
    qk_norm=True,
    moe_num_experts=4,
    moe_top_k=2,
    moe_every_n_layers=3,
)

summary_rows = []
for name, cfg in [("dense", dense_cfg), ("moe", moe_cfg)]:
    summary_rows.append(
        {
            "name": name,
            "params_m": count_parameters(ResearchTransformer(cfg)) / 1e6,
            "kv_cache_mb@2k": estimate_kv_cache_bytes(cfg, batch_size=1, seq_len=2048) / 1024**2,
            "n_layers": cfg.n_layers,
            "n_heads": cfg.n_heads,
            "n_kv_heads": cfg.n_kv_heads,
            "moe_num_experts": cfg.moe_num_experts,
        }
    )

print(json.dumps(summary_rows, indent=2))

fig = go.Figure()
fig.add_bar(name="params (M)", x=[row["name"] for row in summary_rows], y=[row["params_m"] for row in summary_rows])
fig.add_bar(name="KV cache MB @2k", x=[row["name"] for row in summary_rows], y=[row["kv_cache_mb@2k"] for row in summary_rows])
fig.update_layout(barmode="group", title="Dense vs optional MoE candidate scale", template="plotly_white")
fig.show()

In [ ]:
# Symbolic systems sanity checks using sympy

L, T, H_kv, d_h, B = sp.symbols("L T H_kv d_h B", positive=True)
kv_bytes_expr = 2 * B * L * T * H_kv * d_h
flop_proxy_expr = B * T * L * (H_kv * d_h + H_kv * d_h)

print("KV-cache bytes expression:")
display(kv_bytes_expr)
print("Very rough attention-state growth proxy:")
display(flop_proxy_expr)

In [ ]:
# Tiny smoke tests for the dense and MoE stacks

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for name, cfg in [("dense", dense_cfg), ("moe", moe_cfg)]:
    model = ResearchTransformer(cfg).to(device)
    x = torch.randint(0, cfg.vocab_size, (2, 128), device=device)
    y = torch.randint(0, cfg.vocab_size, (2, 128), device=device)
    out = model(x, targets=y, capture_diagnostics=True)
    print("=" * 100)
    print(name)
    print("logits shape:", tuple(out["logits"].shape))
    print("loss:", float(out["loss"]))
    print("aux_loss:", float(out["aux_loss"]))
    print("last block diagnostics:")
    print(json.dumps(out["diagnostics"][-1], indent=2))

In [ ]:
# Stage plan and launch bundles

PHASE_DATA_DIR = PROJECT_ROOT / "artifacts_500m_kb" / "data" / "phase_tokens"
base_train = PHASE_DATA_DIR / "base_train_tokens.npy"
base_val = PHASE_DATA_DIR / "base_val_tokens.npy"
adapt_train = PHASE_DATA_DIR / "adapt_train_tokens.npy"
adapt_val = PHASE_DATA_DIR / "adapt_val_tokens.npy"

stage_plan = build_stage_plan(str(base_train), str(base_val), str(adapt_train), str(adapt_val))
for stage in stage_plan:
    print(stage)

base_train_cfg = TrainConfig(run_name="small_frontier_dense", stage_name=stage_plan[0].name)
moe_train_cfg = TrainConfig(run_name="small_frontier_moe", stage_name=stage_plan[0].name)

dense_bundle_path = write_launch_bundle(LAUNCH_ROOT / "dense_base_bundle.json", dense_cfg, base_train_cfg, stage_plan[0])
moe_bundle_path = write_launch_bundle(LAUNCH_ROOT / "moe_base_bundle.json", moe_cfg, moe_train_cfg, stage_plan[0])

print("Dense launch bundle:", dense_bundle_path)
print("MoE launch bundle:", moe_bundle_path)

In [ ]:
# Torchrun launch previews for the new static worker

launch_dense = FsdpLaunchConfig(nproc_per_node=3).as_command(dense_bundle_path)
launch_moe = FsdpLaunchConfig(nproc_per_node=3).as_command(moe_bundle_path)

print("Dense launch command:")
print(" ".join(launch_dense))
print("\nMoE launch command:")
print(" ".join(launch_moe))


In [ ]:
# Metrics and dashboard loader

def candidate_metric_paths():
    paths = []
    for cfg_name in ["small_frontier_dense", "small_frontier_moe"]:
        for stage in ["base_pretrain", "domain_adapt"]:
            paths.append(ARTIFACT_ROOT / cfg_name / stage / "metrics.jsonl")
            paths.append(PROJECT_ROOT / "artifacts_500m_kb" / "runs" / f"kb_500m_fsdp_{'base' if stage == 'base_pretrain' else 'adapt'}" / "metrics.jsonl")
    return paths

available = [path for path in candidate_metric_paths() if path.exists()]
print("Available metric files:")
for path in available:
    print("-", path)

In [ ]:
# Interactive Plotly dashboard tabs, if metrics exist

available_metric_files = [path for path in candidate_metric_paths() if path.exists()]

if not available_metric_files:
    print("No metrics files found yet. Run a stage first, then re-open this cell.")
else:
    figures = []
    titles = []
    for metric_path in available_metric_files[:2]:
        rows = load_metrics_rows(metric_path)
        if not rows:
            continue
        titles.append(metric_path.parent.name)
        figures.append(build_training_overview(rows))
        if any("router_entropy" in row for row in rows):
            titles.append(metric_path.parent.name + "_moe")
            figures.append(build_moe_dashboard(rows))

    children = [go.FigureWidget(fig) for fig in figures]
    tab = widgets.Tab(children=children)
    for idx, title in enumerate(titles):
        tab.set_title(idx, title[:18])
    display(tab)

In [ ]:
# Attention and residual diagnostics from a tiny probe model

probe_model = ResearchTransformer(dense_cfg).to(device)
probe_ids = torch.randint(0, dense_cfg.vocab_size, (1, 96), device=device)
probe_out = probe_model(probe_ids, capture_diagnostics=True)
probe_fig = build_attention_probe_figure(probe_out["diagnostics"])
probe_fig.show()

In [ ]:
# Next-token probability dashboard on the probe model

prompt_ids = torch.randint(0, dense_cfg.vocab_size, (1, 64), device=device)
probe_out = probe_model(prompt_ids, capture_diagnostics=False)
probs = torch.softmax(probe_out["logits"][0, -1], dim=-1)
values, indices = torch.topk(probs, k=15)
labels = [str(int(index.item())) for index in indices]
next_token_fig = build_next_token_figure(labels, values.detach().cpu().tolist())
next_token_fig.show()

In [ ]:
# Curated post-training set and reward-guided utilities

instruction_examples = build_repo_instruction_set(PROJECT_ROOT)
print(f"Loaded {len(instruction_examples)} instruction examples.")
for example in instruction_examples[:4]:
    print("=" * 100)
    print("PROMPT:", example.prompt)
    print("RESPONSE:", example.response)
    print("SOURCE:", example.source)

reward_demo = outcome_reward(
    candidate="Grouped-query attention reduces KV-cache pressure because keys and values are shared across groups of query heads.",
    reference="Grouped-query attention reduces KV-cache growth by sharing keys and values across multiple query heads.",
    required_keywords=["kv-cache", "shared", "heads"],
)
print("\nReward demo:", reward_demo)


In [ ]:
# Simple calibration example for verifier-style evaluation summaries

calibration = calibration_bins(
    confidences=[0.2, 0.35, 0.51, 0.68, 0.74, 0.82, 0.93],
    correct=[0, 0, 1, 1, 1, 0, 1],
    num_bins=5,
)
print(json.dumps(calibration, indent=2))

## Recommended Run Order

1. Build or verify the token arrays under `artifacts_500m_kb/data/phase_tokens/`.
2. Run the dense smoke-test and dense launch bundle cells.
3. Launch dense base pretraining with the static worker.
4. Re-open the dashboard cells once `metrics.jsonl` exists.
5. Run domain adaptation after the dense base checkpoint is stable.
6. Use the SFT and reward-guided cells after the dense model has a clean baseline.
7. Only then run the MoE branch and compare against dense under matched active compute.